<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 02
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Khai thác và trực quan hóa dữ liệu bằng Tableau"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TỪ WORLD BANK
</div>

## **1. NHẬP DỮ LIỆU VÀ KIỂM TRA TỔNG QUAN**

### **1.1. Mục tiêu**

Bước đầu tiên trong quy trình tiền xử lý là đọc dữ liệu thô từ World Development Indicators, kiểm tra cấu trúc file, phát hiện các dòng thừa, kiểm tra encoding, kiểm tra phạm vi năm, số lượng quốc gia/vùng lãnh thổ và số lượng chỉ số phát triển.

Bộ dữ liệu được tải từ World Bank dưới dạng bảng rộng, trong đó mỗi dòng tương ứng với một cặp **quốc gia/vùng lãnh thổ - chỉ số**, còn các cột năm từ `2000 [YR2000]` đến `2022 [YR2022]` chứa giá trị quan sát theo thời gian.

### **1.2. Ghi chú ban đầu về file dữ liệu**

Qua kiểm tra sơ bộ:

* `Data.csv` là file dữ liệu chính, chứa các giá trị chỉ số theo quốc gia/vùng lãnh thổ và theo năm.
* `Metadata.xlsx` là file metadata tải riêng từ World Bank, gồm nhiều sheet:
    * `Series - Metadata`: metadata mô tả các chỉ số.
    * `Country - Metadata`: metadata mô tả quốc gia/vùng lãnh thổ.
    * `Country-Series - Metadata`: ghi chú theo từng cặp quốc gia - chỉ số.
    * `FootNote`: ghi chú theo từng quốc gia - chỉ số - năm.
* Trong phạm vi tiền xử lý chính, nhóm sử dụng `Series - Metadata` để giải thích ý nghĩa chỉ số và `Country - Metadata` để bổ sung thông tin khu vực, nhóm thu nhập.
* Dữ liệu hiện có phạm vi năm **2000–2022**.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1: IMPORT THƯ VIỆN VÀ THIẾT LẬP ĐƯỜNG DẪN
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Xác định thư mục gốc project LAB02
# Notebook đang nằm trong LAB02/notebooks/
PROJECT_ROOT = Path.cwd()

# Nếu đang chạy notebook từ thư mục notebooks thì đi lên 1 cấp
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH = RAW_DATA_DIR / "Data.csv"
METADATA_PATH = RAW_DATA_DIR / "Metadata.xlsx"

OUTPUT_DIR = PROCESSED_DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục project        :", PROJECT_ROOT)
print("Đường dẫn Data.csv     :", DATA_PATH)
print("Đường dẫn Metadata.xlsx:", METADATA_PATH)
print("Thư mục xuất dữ liệu   :", OUTPUT_DIR)

# Kiểm tra file có tồn tại không
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy Data.csv tại: {DATA_PATH}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy Metadata.xlsx tại: {METADATA_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1.1: ĐỌC DỮ LIỆU CHÍNH VÀ METADATA
# ─────────────────────────────────────────────────────────────

def read_csv_with_fallback(path, encodings=("utf-8-sig", "utf-8", "cp1252", "latin1")):
    """
    Đọc file CSV với nhiều encoding khác nhau.
    File Data.csv của World Bank thường đọc tốt bằng utf-8-sig,
    nhưng vẫn dùng fallback để tránh lỗi encoding.
    """
    last_error = None
    
    for enc in encodings:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            print(f"Đọc thành công {path.name} với encoding: {enc}")
            return df, enc
        except UnicodeDecodeError as e:
            last_error = e
    
    raise UnicodeDecodeError(
        "Không đọc được file với các encoding đã thử.",
        b"",
        0,
        1,
        str(last_error)
    )

# Đọc dữ liệu chính
df_data_raw, data_encoding = read_csv_with_fallback(DATA_PATH)

# Đọc metadata từ file Excel tải riêng từ World Bank
metadata_series_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Series - Metadata",
    dtype=str
)

metadata_country_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Country - Metadata",
    dtype=str
)

metadata_country_series_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Country-Series - Metadata",
    dtype=str
)

metadata_footnote_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="FootNote",
    dtype=str
)

print("\n--- Kích thước dữ liệu thô")
print(f"Data.csv                    : {df_data_raw.shape}")
print(f"Series - Metadata           : {metadata_series_raw.shape}")
print(f"Country - Metadata          : {metadata_country_raw.shape}")
print(f"Country-Series - Metadata   : {metadata_country_series_raw.shape}")
print(f"FootNote                    : {metadata_footnote_raw.shape}")

print("\n--- 5 dòng đầu của Data.csv")
display(df_data_raw.head())

print("\n--- 5 dòng cuối của Data.csv")
display(df_data_raw.tail(5))

print("\n--- 5 dòng đầu của Series - Metadata")
display(metadata_series_raw.head())

print("\n--- 5 dòng đầu của Country - Metadata")
display(metadata_country_raw.head())

### **1.3. Kiểm tra cấu trúc dữ liệu thô**

File `Data.csv` đang ở dạng bảng rộng:

* 4 cột định danh chính:
    * `Country Name`
    * `Country Code`
    * `Series Name`
    * `Series Code`
* Các cột năm:
    * `2000 [YR2000]`
    * ...
    * `2022 [YR2022]`

Trong dữ liệu gốc, World Bank dùng ký hiệu `..` để biểu diễn giá trị thiếu. Các giá trị này cần được chuyển thành `NaN` trước khi ép kiểu số.

Ngoài ra, cuối file có các dòng footer không phải quan sát dữ liệu thật, nên cần loại bỏ.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1.2: KIỂM TRA CẤU TRÚC VÀ PHÁT HIỆN CỘT NĂM
# ─────────────────────────────────────────────────────────────

ID_COLS_RAW = ["Country Name", "Country Code", "Series Name", "Series Code"]

# Tự động nhận diện các cột năm dạng "2000 [YR2000]"
YEAR_COLS_RAW = [
    col for col in df_data_raw.columns
    if re.match(r"^\d{4}\s+\[YR\d{4}\]$", str(col))
]

print("--- Danh sách cột định danh")
print(ID_COLS_RAW)

print("\n--- Số lượng cột năm")
print(len(YEAR_COLS_RAW))

print("\n--- Phạm vi năm trong file")
years_detected = [int(re.search(r"\d{4}", col).group()) for col in YEAR_COLS_RAW]
print(min(years_detected), "→", max(years_detected))

print("\n--- Danh sách cột năm")
print(YEAR_COLS_RAW)

print("\n--- Kiểm tra các cột bắt buộc")
missing_required_cols = [col for col in ID_COLS_RAW if col not in df_data_raw.columns]
if missing_required_cols:
    print("Thiếu cột bắt buộc:", missing_required_cols)
else:
    print("Đủ các cột định danh bắt buộc.")

## **2. LÀM SẠCH CẤU TRÚC DỮ LIỆU**

### **2.1. Loại bỏ dòng thừa và chuẩn hóa kiểu dữ liệu**

Dữ liệu thô có một số dòng cuối file không phải quan sát dữ liệu, bao gồm dòng trống và dòng ghi chú từ World Bank. Các dòng này không có đầy đủ `Country Code`, `Series Name`, `Series Code`, do đó có thể loại bỏ an toàn.

Sau đó, các cột năm được xử lý theo quy tắc:

* Chuyển ký hiệu `..` thành `NaN`.
* Ép kiểu các cột năm về dạng số thực.
* Giữ nguyên giá trị thiếu thay vì tự ý điền trung bình hoặc nội suy.

Đối với dữ liệu phát triển quốc gia theo năm, việc tự ý điền thiếu có thể làm sai lệch xu hướng thật. Vì vậy, nhóm chỉ chuẩn hóa kiểu dữ liệu và tạo thêm bảng thống kê missing để phục vụ báo cáo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 2: LÀM SẠCH DATA.CSV
# ─────────────────────────────────────────────────────────────

df_wide = df_data_raw.copy()

# 1. Chuẩn hóa khoảng trắng trong tên cột
df_wide.columns = df_wide.columns.str.strip()

# 2. Chuẩn hóa chuỗi rỗng thành NaN
df_wide = df_wide.replace(r"^\s*$", np.nan, regex=True)

# 3. Loại bỏ các dòng không có đủ 4 cột định danh
before_rows = len(df_wide)
df_wide = df_wide.dropna(subset=ID_COLS_RAW, how="any").copy()
after_drop_id_rows = len(df_wide)

# 4. Loại bỏ các dòng footer nếu còn sót
footer_mask = df_wide["Country Name"].astype(str).str.startswith("Data from database", na=False) | \
              df_wide["Country Name"].astype(str).str.startswith("Last Updated", na=False)

df_wide = df_wide[~footer_mask].copy()
after_drop_footer_rows = len(df_wide)

# 5. Chuẩn hóa khoảng trắng trong các cột định danh
for col in ID_COLS_RAW:
    df_wide[col] = df_wide[col].astype(str).str.strip()

# 6. Chuyển ".." thành NaN và ép kiểu số cho các cột năm
for col in YEAR_COLS_RAW:
    df_wide[col] = (
        df_wide[col]
        .replace("..", np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )

print("--- Kết quả làm sạch cấu trúc")
print(f"Số dòng ban đầu                         : {before_rows}")
print(f"Số dòng sau khi drop dòng thiếu định danh: {after_drop_id_rows}")
print(f"Số dòng sau khi drop footer              : {after_drop_footer_rows}")

print("\n--- Kích thước dữ liệu sau làm sạch")
print(df_wide.shape)

print("\n--- Kiểu dữ liệu sau xử lý")
display(df_wide.dtypes.to_frame("dtype").T)

print("\n--- 5 dòng đầu sau làm sạch")
display(df_wide.head())

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 2.1: KIỂM TRA TÍNH TOÀN VẸN CỦA DỮ LIỆU
# ─────────────────────────────────────────────────────────────

n_countries = df_wide["Country Code"].nunique()
n_indicators = df_wide["Series Code"].nunique()
n_years = len(YEAR_COLS_RAW)

expected_rows = n_countries * n_indicators
actual_rows = len(df_wide)

duplicate_keys = df_wide.duplicated(subset=["Country Code", "Series Code"]).sum()
rows_all_year_missing = df_wide[YEAR_COLS_RAW].isna().all(axis=1).sum()

print("=" * 70)
print("KIỂM TRA TỔNG QUAN DỮ LIỆU SAU LÀM SẠCH")
print("=" * 70)

print(f"Số quốc gia/vùng/tổng hợp       : {n_countries}")
print(f"Số chỉ số phát triển             : {n_indicators}")
print(f"Số năm quan sát                  : {n_years}")
print(f"Số dòng kỳ vọng                  : {expected_rows}")
print(f"Số dòng thực tế                  : {actual_rows}")
print(f"Số khóa Country Code + Series Code bị trùng: {duplicate_keys}")
print(f"Số dòng thiếu toàn bộ các năm    : {rows_all_year_missing}")

if expected_rows == actual_rows and duplicate_keys == 0:
    print("\nDữ liệu có cấu trúc nhất quán: mỗi quốc gia/vùng có đủ dòng cho mỗi chỉ số.")
else:
    print("\nCần kiểm tra thêm vì số dòng thực tế không khớp cấu trúc kỳ vọng.")

print("\n--- Danh sách 12 chỉ số trong dữ liệu")
indicator_list = (
    df_wide[["Series Code", "Series Name"]]
    .drop_duplicates()
    .sort_values("Series Code")
    .reset_index(drop=True)
)
display(indicator_list)

## **3. TÁCH VÀ CHUẨN HÓA METADATA**

### **3.1. Metadata chỉ số và metadata quốc gia**
Nhóm sử dụng hai sheet chính:

* `Series - Metadata`: mô tả các chỉ số phát triển, bao gồm mã chỉ số, tên chỉ số, định nghĩa, chủ đề, đơn vị đo, chu kỳ cập nhật, phương pháp tổng hợp và nguồn dữ liệu.
* `Country - Metadata`: mô tả quốc gia/vùng lãnh thổ, bao gồm tên đầy đủ, nhóm thu nhập, khu vực, đơn vị tiền tệ và một số thông tin thống kê bổ sung.

Việc dùng metadata giúp dữ liệu sau tiền xử lý có thêm ngữ cảnh, thuận tiện hơn cho việc xây dựng dashboard trong Tableau và diễn giải kết quả trong báo cáo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 3: CHUẨN HÓA METADATA CHỈ SỐ VÀ METADATA QUỐC GIA
# ─────────────────────────────────────────────────────────────

# 3.1. Chuẩn hóa metadata chỉ số
indicator_metadata = metadata_series_raw.copy()
indicator_metadata.columns = indicator_metadata.columns.str.strip()
indicator_metadata = indicator_metadata.replace(r"^\s*$", np.nan, regex=True)

rename_indicator_cols = {
    "Code": "indicator_code",
    "Indicator Name": "indicator_name_en",
    "Long definition": "long_definition",
    "Source": "source",
    "Topic": "topic",
    "Dataset": "dataset",
    "Unit of measure": "unit_of_measure",
    "Periodicity": "periodicity",
    "Reference period": "reference_period",
    "Aggregation method": "aggregation_method",
    "Statistical concept and methodology": "methodology",
    "Development relevance": "development_relevance",
    "Limitations and exceptions": "limitations",
    "Other notes": "other_notes",
    "License URL": "license_url",
    "License Type": "license_type"
}

indicator_metadata = indicator_metadata.rename(columns=rename_indicator_cols)

valid_indicator_codes = set(df_wide["Series Code"].unique())

indicator_metadata = indicator_metadata[
    indicator_metadata["indicator_code"].isin(valid_indicator_codes)
].copy()

indicator_metadata = indicator_metadata.drop_duplicates(subset=["indicator_code"])

print("--- Metadata chỉ số sau khi chuẩn hóa")
print(indicator_metadata.shape)

display(indicator_metadata.head())


# 3.2. Chuẩn hóa metadata quốc gia/vùng lãnh thổ
country_metadata = metadata_country_raw.copy()
country_metadata.columns = country_metadata.columns.str.strip()
country_metadata = country_metadata.replace(r"^\s*$", np.nan, regex=True)

rename_country_cols = {
    "Code": "country_code",
    "Long Name": "country_long_name",
    "Income Group": "income_group",
    "Region": "region",
    "Lending category": "lending_category",
    "Other groups": "other_groups",
    "Currency Unit": "currency_unit",
    "Latest population census": "latest_population_census",
    "Latest household survey": "latest_household_survey",
    "Special Notes": "country_special_notes",
    "Table Name": "country_table_name",
    "Short Name": "country_short_name",
    "2-alpha code": "country_alpha2_code",
    "WB-2 code": "country_wb2_code"
}

country_metadata = country_metadata.rename(columns=rename_country_cols)

country_keep_cols = [
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_table_name",
    "region",
    "income_group",
    "lending_category",
    "other_groups",
    "currency_unit",
    "country_alpha2_code",
    "country_wb2_code",
    "latest_population_census",
    "latest_household_survey",
    "country_special_notes"
]

country_keep_cols = [col for col in country_keep_cols if col in country_metadata.columns]

country_metadata = (
    country_metadata[country_keep_cols]
    .drop_duplicates(subset=["country_code"])
    .reset_index(drop=True)
)

print("\n--- Metadata quốc gia/vùng lãnh thổ sau khi chuẩn hóa")
print(country_metadata.shape)

display(country_metadata.head())

### **3.2. Việt hóa tên chỉ số và phân nhóm phân tích**

Để thuận tiện khi trực quan hóa bằng Tableau và viết báo cáo, nhóm tạo thêm các trường mô tả bằng tiếng Việt cho từng chỉ số.

Các chỉ số được chia thành 4 nhóm chủ đề chính:

* **Kinh tế:** GDP bình quân đầu người, tăng trưởng GDP bình quân đầu người, tỷ lệ nghèo.
* **Sức khỏe:** Tuổi thọ, chi tiêu y tế, tử vong trẻ em dưới 5 tuổi.
* **Môi trường:** Phát thải CO2 bình quân đầu người, tỷ trọng năng lượng tái tạo.
* **Giáo dục:** Tỷ lệ nhập học các cấp và chỉ số cân bằng giới trong giáo dục.

Các trường này không làm thay đổi dữ liệu gốc mà chỉ bổ sung nhãn diễn giải phục vụ dashboard.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 3.2: VIỆT HÓA TÊN CHỈ SỐ VÀ PHÂN NHÓM CHỦ ĐỀ
# ─────────────────────────────────────────────────────────────

INDICATOR_INFO_VI = {
    "NY.GDP.PCAP.KD": {
        "indicator_name_vi": "GDP bình quân đầu người (USD cố định 2015)",
        "indicator_group": "Kinh tế",
        "unit_vi": "USD cố định 2015",
        "short_name": "GDP/người",
        "tableau_field_name": "gdp_per_capita"
    },
    "NY.GDP.PCAP.KD.ZG": {
        "indicator_name_vi": "Tăng trưởng GDP bình quân đầu người (%)",
        "indicator_group": "Kinh tế",
        "unit_vi": "%",
        "short_name": "Tăng trưởng GDP/người",
        "tableau_field_name": "gdp_per_capita_growth_pct"
    },
    "SI.POV.NAHC": {
        "indicator_name_vi": "Tỷ lệ nghèo theo chuẩn quốc gia (%)",
        "indicator_group": "Kinh tế - Xã hội",
        "unit_vi": "%",
        "short_name": "Tỷ lệ nghèo",
        "tableau_field_name": "poverty_headcount_pct"
    },
    "SP.DYN.LE00.IN": {
        "indicator_name_vi": "Tuổi thọ trung bình khi sinh (năm)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "Năm",
        "short_name": "Tuổi thọ",
        "tableau_field_name": "life_expectancy_years"
    },
    "SH.XPD.CHEX.PC.CD": {
        "indicator_name_vi": "Chi tiêu y tế bình quân đầu người (USD hiện hành)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "USD hiện hành",
        "short_name": "Chi tiêu y tế/người",
        "tableau_field_name": "health_expenditure_pc_usd"
    },
    "SH.DYN.MORT": {
        "indicator_name_vi": "Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh sống)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "Trên 1.000 ca sinh sống",
        "short_name": "Tử vong dưới 5 tuổi",
        "tableau_field_name": "under5_mortality_per_1000"
    },
    "EN.GHG.CO2.PC.CE.AR5": {
        "indicator_name_vi": "Phát thải CO2 bình quân đầu người (tCO2e/người)",
        "indicator_group": "Môi trường",
        "unit_vi": "tCO2e/người",
        "short_name": "CO2/người",
        "tableau_field_name": "co2_pc_tco2e"
    },
    "EG.FEC.RNEW.ZS": {
        "indicator_name_vi": "Tỷ trọng năng lượng tái tạo trong tiêu thụ năng lượng cuối cùng (%)",
        "indicator_group": "Môi trường",
        "unit_vi": "%",
        "short_name": "Năng lượng tái tạo",
        "tableau_field_name": "renewable_energy_pct"
    },
    "SE.PRM.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học tiểu học, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học tiểu học",
        "tableau_field_name": "school_primary_gross_pct"
    },
    "SE.SEC.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học trung học, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học trung học",
        "tableau_field_name": "school_secondary_gross_pct"
    },
    "SE.TER.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học bậc cao",
        "tableau_field_name": "school_tertiary_gross_pct"
    },
    "SE.ENR.PRSC.FM.ZS": {
        "indicator_name_vi": "Chỉ số cân bằng giới trong nhập học tiểu học và trung học (GPI)",
        "indicator_group": "Giáo dục",
        "unit_vi": "Tỷ lệ nữ/nam",
        "short_name": "GPI giáo dục",
        "tableau_field_name": "education_gpi"
    }
}

indicator_info_vi = (
    pd.DataFrame.from_dict(INDICATOR_INFO_VI, orient="index")
    .reset_index()
    .rename(columns={"index": "indicator_code"})
)

indicator_metadata = indicator_metadata.merge(
    indicator_info_vi,
    on="indicator_code",
    how="left"
)

# Nếu metadata không có tên tiếng Anh vì đọc lỗi, lấy từ Data.csv để bổ sung
series_name_lookup = (
    df_wide[["Series Code", "Series Name"]]
    .drop_duplicates()
    .rename(columns={
        "Series Code": "indicator_code",
        "Series Name": "indicator_name_from_data"
    })
)

indicator_metadata = indicator_metadata.merge(
    series_name_lookup,
    on="indicator_code",
    how="left"
)

indicator_metadata["indicator_name_en"] = indicator_metadata["indicator_name_en"].fillna(
    indicator_metadata["indicator_name_from_data"]
)

indicator_metadata = indicator_metadata.drop(columns=["indicator_name_from_data"], errors="ignore")

print("--- Metadata sau khi bổ sung tên tiếng Việt và nhóm chỉ số")
display(indicator_metadata[[
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "indicator_group",
    "unit_vi",
    "short_name",
    "tableau_field_name"
]])

print("\n--- Kiểm tra số lượng chỉ số sau khi bổ sung metadata tiếng Việt")
print(indicator_metadata["indicator_code"].nunique(), "chỉ số")

print("\n--- Các chỉ số chưa có tên tiếng Việt nếu có")
missing_vi_name = indicator_metadata[indicator_metadata["indicator_name_vi"].isna()]
display(missing_vi_name[["indicator_code", "indicator_name_en"]])

## **4. CHUYỂN DỮ LIỆU TỪ DẠNG RỘNG SANG DẠNG DÀI**

### **4.1. Lý do chuyển đổi**

Dữ liệu tải từ World Bank đang ở dạng rộng, mỗi năm là một cột riêng. Dạng này thuận tiện để đọc thủ công nhưng chưa tối ưu cho Tableau.

Nhóm chuyển dữ liệu sang dạng dài với cấu trúc:

* `country_name`
* `country_code`
* `indicator_code`
* `indicator_name_en`
* `indicator_name_vi`
* `indicator_group`
* `year`
* `value`

Dạng dài phù hợp hơn cho:

* Vẽ biểu đồ chuỗi thời gian.
* Dùng filter theo năm, quốc gia, nhóm chỉ số.
* Xây dựng dashboard có selector/dropdown trong Tableau.
* So sánh linh hoạt nhiều chỉ số theo thời gian.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 4: CHUYỂN DỮ LIỆU WIDE → LONG
# ─────────────────────────────────────────────────────────────

df_long = df_wide.melt(
    id_vars=ID_COLS_RAW,
    value_vars=YEAR_COLS_RAW,
    var_name="year_raw",
    value_name="value"
)

# Trích năm từ chuỗi "2000 [YR2000]"
df_long["year"] = df_long["year_raw"].str.extract(r"(\d{4})").astype(int)

# Đổi tên cột sang dạng snake_case
df_long = df_long.rename(columns={
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Series Name": "indicator_name_en_raw",
    "Series Code": "indicator_code"
})

# Join metadata chỉ số
metadata_join_cols = [
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "indicator_group",
    "unit_vi",
    "short_name",
    "tableau_field_name",
    "long_definition",
    "topic",
    "source",
    "unit_of_measure",
    "periodicity",
    "reference_period",
    "aggregation_method"
]

metadata_join_cols = [col for col in metadata_join_cols if col in indicator_metadata.columns]

df_long = df_long.merge(
    indicator_metadata[metadata_join_cols],
    on="indicator_code",
    how="left"
)

# Nếu tên tiếng Anh từ metadata thiếu thì dùng tên trong Data.csv
df_long["indicator_name_en"] = df_long["indicator_name_en"].fillna(
    df_long["indicator_name_en_raw"]
)

# Join metadata quốc gia/vùng lãnh thổ
country_join_cols = [
    "country_code",
    "country_long_name",
    "country_short_name",
    "region",
    "income_group",
    "currency_unit"
]

country_join_cols = [col for col in country_join_cols if col in country_metadata.columns]

df_long = df_long.merge(
    country_metadata[country_join_cols],
    on="country_code",
    how="left"
)

# Tạo cột ngày để Tableau nhận diện time-series dễ hơn
df_long["year_date"] = pd.to_datetime(df_long["year"].astype(str) + "-01-01")

# Tạo giai đoạn để hỗ trợ filter/nhóm trong Tableau
def period_group(year):
    if 2000 <= year <= 2004:
        return "2000–2004"
    if 2005 <= year <= 2009:
        return "2005–2009"
    if 2010 <= year <= 2014:
        return "2010–2014"
    if 2015 <= year <= 2019:
        return "2015–2019"
    if 2020 <= year <= 2022:
        return "2020–2022"
    return "Khác"

df_long["period_group"] = df_long["year"].apply(period_group)

# Cờ dữ liệu thiếu
df_long["is_missing"] = df_long["value"].isna()
df_long["data_status"] = np.where(df_long["is_missing"], "Thiếu dữ liệu", "Có dữ liệu")

print("--- Kích thước dữ liệu dạng dài")
print(df_long.shape)

print("\n--- 5 dòng đầu")
display(df_long.head())

print("\n--- Tỉ lệ missing toàn bộ bảng long")
print(f"{df_long['value'].isna().mean() * 100:.2f}%")

print("\n--- Các mã quốc gia có trong Data.csv nhưng không có trong Country - Metadata")
missing_country_metadata = (
    df_long[df_long["country_long_name"].isna()][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_code")
)

display(missing_country_metadata)

## **5. NHẬN DIỆN QUỐC GIA THẬT VÀ NHÓM TỔNG HỢP**

### **5.1. Vấn đề về đơn vị quan sát**

Dữ liệu WDI không chỉ bao gồm quốc gia/vùng lãnh thổ riêng lẻ mà còn có các nhóm tổng hợp như:

* `World`
* `High income`
* `Low income`
* `East Asia & Pacific`
* `European Union`
* `OECD members`
* `Sub-Saharan Africa`

Các nhóm tổng hợp này hữu ích khi so sánh bối cảnh chung, nhưng nếu phân tích xếp hạng quốc gia thì cần loại ra để tránh sai lệch.

Vì vậy, nhóm không xóa các dòng này khỏi dữ liệu chính, mà tạo thêm cột `is_aggregate` và `country_type` để có thể lọc linh hoạt trong Tableau.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 5: ĐÁNH DẤU NHÓM TỔNG HỢP CỦA WORLD BANK
# ─────────────────────────────────────────────────────────────

# Danh sách mã tổng hợp phổ biến của World Bank trong WDI
# Không xóa khỏi dữ liệu chính, chỉ đánh dấu để lọc trong Tableau.
WB_AGGREGATE_CODES = {
    "AFE", "AFW", "ARB", "CSS", "CEB",
    "EAR", "EAS", "EAP", "TEA",
    "EMU", "ECS", "ECA", "TEC", "EUU",
    "FCS", "HPC",
    "HIC", "IBD", "IBT", "IDB", "IDX", "IDA",
    "LTE", "LCN", "LAC", "TLA", "LDC",
    "LMY", "LIC", "LMC", "MEA", "MNA", "TMN",
    "MIC", "NAC", "INX", "OED", "OSS", "PSS",
    "PST", "PRE", "SST", "SAS", "TSA",
    "SSF", "SSA", "TSS", "UMC", "WLD"
}

df_long["is_aggregate"] = df_long["country_code"].isin(WB_AGGREGATE_CODES)

df_long["country_type"] = np.where(
    df_long["is_aggregate"],
    "Nhóm tổng hợp/khu vực",
    "Quốc gia/vùng lãnh thổ"
)

print("--- Số lượng đơn vị theo loại")
display(
    df_long[["country_name", "country_code", "country_type"]]
    .drop_duplicates()
    .groupby("country_type")
    .size()
    .reset_index(name="so_luong")
)

print("\n--- Một số nhóm tổng hợp trong dữ liệu")
display(
    df_long[df_long["is_aggregate"]][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_name")
    .head(30)
)

print("\n--- Một số quốc gia/vùng lãnh thổ thật")
display(
    df_long[~df_long["is_aggregate"]][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_name")
    .head(30)
)

## **6. PHÂN TÍCH CHẤT LƯỢNG DỮ LIỆU**

### **6.1. Kiểm tra dữ liệu thiếu**

Dữ liệu World Development Indicators có mức độ thiếu khác nhau giữa các chỉ số. Đây là đặc điểm bình thường của dữ liệu phát triển quốc tế vì không phải quốc gia nào cũng công bố đầy đủ mọi chỉ số qua từng năm.

Nguyên tắc xử lý của nhóm:

* Không tự ý điền trung bình cho các chỉ số theo quốc gia.
* Không nội suy tuyến tính trên toàn bộ dữ liệu vì có thể tạo xu hướng giả.
* Giữ bảng đầy đủ có cả missing để kiểm tra chất lượng.
* Xuất bảng sạch không missing để đưa vào Tableau khi cần vẽ biểu đồ.
* Với các chỉ số thiếu nhiều như tỷ lệ nghèo, khi phân tích cần ghi rõ số lượng quan sát và tránh kết luận quá rộng.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6: THỐNG KÊ MISSING VALUE
# ─────────────────────────────────────────────────────────────

# 1. Missing theo chỉ số
missing_by_indicator = (
    df_long
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "indicator_group"
    ])
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
    .sort_values("missing_pct", ascending=False)
)

missing_by_indicator["missing_pct"] = missing_by_indicator["missing_pct"].round(2)

print("--- Tỉ lệ missing theo chỉ số")
display(missing_by_indicator)

# 2. Missing theo năm
missing_by_year = (
    df_long
    .groupby("year")
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
)

missing_by_year["missing_pct"] = missing_by_year["missing_pct"].round(2)

print("\n--- Tỉ lệ missing theo năm")
display(missing_by_year)

# 3. Missing theo chỉ số và năm
missing_by_indicator_year = (
    df_long
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "year"
    ])
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
)

missing_by_indicator_year["missing_pct"] = missing_by_indicator_year["missing_pct"].round(2)

print("\n--- Missing theo chỉ số và năm")
display(missing_by_indicator_year.head(20))

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6.1: KIỂM TRA CÁC DÒNG THIẾU TOÀN BỘ THEO QUỐC GIA - CHỈ SỐ
# ─────────────────────────────────────────────────────────────

# Một dòng trong df_wide tương ứng với một cặp country_code - indicator_code.
# Nếu toàn bộ 23 năm đều thiếu thì dòng này không có giá trị phân tích trực tiếp.
rows_all_missing = df_wide.copy()
rows_all_missing["all_years_missing"] = rows_all_missing[YEAR_COLS_RAW].isna().all(axis=1)

all_missing_summary = (
    rows_all_missing[rows_all_missing["all_years_missing"]]
    [["Country Name", "Country Code", "Series Code", "Series Name"]]
    .rename(columns={
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Series Code": "indicator_code",
        "Series Name": "indicator_name_en"
    })
    .sort_values(["indicator_code", "country_name"])
    .reset_index(drop=True)
)

print("--- Số dòng country-indicator thiếu toàn bộ các năm")
print(len(all_missing_summary))

print("\n--- Ví dụ các dòng thiếu toàn bộ")
display(all_missing_summary.head(30))

# Thống kê số indicator bị thiếu toàn bộ theo từng quốc gia
country_coverage_summary = (
    rows_all_missing
    .groupby(["Country Name", "Country Code"])
    .agg(
        total_indicators=("Series Code", "count"),
        indicators_all_years_missing=("all_years_missing", "sum")
    )
    .reset_index()
    .rename(columns={
        "Country Name": "country_name",
        "Country Code": "country_code"
    })
)

country_coverage_summary["available_indicator_ratio"] = (
    1 - country_coverage_summary["indicators_all_years_missing"] / country_coverage_summary["total_indicators"]
).round(3)

country_coverage_summary = country_coverage_summary.sort_values(
    ["indicators_all_years_missing", "country_name"],
    ascending=[False, True]
)

print("\n--- Quốc gia/vùng có nhiều chỉ số thiếu toàn bộ nhất")
display(country_coverage_summary.head(20))

## **7. TẠO BẢNG DỮ LIỆU SẠCH CHO TABLEAU**

### **7.1. Bảng long sạch**

Bảng `df_long_clean` là bảng chính khuyến nghị dùng trong Tableau. Bảng này:

* Ở dạng dài.
* Có một dòng cho mỗi tổ hợp quốc gia/vùng - chỉ số - năm.
* Chỉ giữ các dòng có giá trị quan sát thật.
* Có đầy đủ nhãn tiếng Việt, nhóm chỉ số, đơn vị đo và cờ đánh dấu nhóm tổng hợp.

Bảng này phù hợp cho hầu hết dashboard như:

* Biểu đồ đường theo thời gian.
* Biểu đồ bản đồ.
* Biểu đồ cột so sánh quốc gia.
* Biểu đồ heatmap theo năm và chỉ số.
* Dashboard có filter theo quốc gia, năm, nhóm chỉ số, loại đơn vị quan sát.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 7: TẠO BẢNG LONG SẠCH
# ─────────────────────────────────────────────────────────────

df_long_clean = df_long.dropna(subset=["value"]).copy()

# Sắp xếp lại cột cho dễ đọc và dễ connect Tableau
long_cols_order = [
    "country_name",
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_type",
    "is_aggregate",
    "region",
    "income_group",
    "currency_unit",
    "indicator_group",
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "short_name",
    "unit_vi",
    "unit_of_measure",
    "topic",
    "source",
    "periodicity",
    "reference_period",
    "aggregation_method",
    "year",
    "year_date",
    "period_group",
    "value",
    "data_status"
]

long_cols_order = [col for col in long_cols_order if col in df_long_clean.columns]
df_long_clean = df_long_clean[long_cols_order].sort_values(
    ["country_name", "indicator_code", "year"]
).reset_index(drop=True)

print("--- Kích thước bảng long đầy đủ, gồm cả missing")
print(df_long.shape)

print("\n--- Kích thước bảng long sạch, đã bỏ missing")
print(df_long_clean.shape)

print("\n--- Tỉ lệ dữ liệu còn lại sau khi bỏ missing")
print(f"{len(df_long_clean) / len(df_long) * 100:.2f}%")

print("\n--- 10 dòng đầu của bảng long sạch")
display(df_long_clean.head(10))

print("\n--- Kiểm tra trùng khóa country_code + indicator_code + year")
long_duplicate_keys = df_long_clean.duplicated(
    subset=["country_code", "indicator_code", "year"]
).sum()

print("Số khóa bị trùng:", long_duplicate_keys)

### **7.2. Bảng wide theo chỉ số**

Ngoài bảng long, nhóm tạo thêm bảng wide theo chỉ số, trong đó mỗi dòng tương ứng với một quốc gia/vùng trong một năm, còn mỗi chỉ số là một cột riêng.

Bảng này phù hợp cho các biểu đồ cần so sánh quan hệ giữa nhiều chỉ số, ví dụ:

* Scatter plot giữa GDP bình quân đầu người và tuổi thọ.
* Scatter plot giữa chi tiêu y tế và tử vong trẻ em.
* So sánh phát thải CO2 và tỷ trọng năng lượng tái tạo.
* Phân tích tương quan giữa giáo dục và kinh tế.

Lưu ý: bảng wide có thể có nhiều giá trị thiếu hơn ở cấp dòng vì không phải quốc gia nào cũng có đủ 12 chỉ số trong cùng một năm.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 7.2: TẠO BẢNG WIDE THEO CHỈ SỐ
# ─────────────────────────────────────────────────────────────

# Lookup tên cột ngắn cho từng indicator
field_name_lookup = indicator_info_vi.set_index("indicator_code")["tableau_field_name"].to_dict()

df_wide_for_tableau = (
    df_long_clean
    .pivot_table(
        index=[
            "country_name",
            "country_code",
            "country_long_name",
            "country_short_name",
            "country_type",
            "is_aggregate",
            "region",
            "income_group",
            "currency_unit",
            "year",
            "year_date",
            "period_group"
        ],
        columns="indicator_code",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Đổi mã WDI thành tên cột dễ dùng trong Tableau
df_wide_for_tableau = df_wide_for_tableau.rename(columns=field_name_lookup)

# Xóa tên trục columns do pivot_table tạo ra
df_wide_for_tableau.columns.name = None

# Sắp xếp cột
base_cols = [
    "country_name",
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_type",
    "is_aggregate",
    "region",
    "income_group",
    "currency_unit",
    "year",
    "year_date",
    "period_group"
]

indicator_wide_cols = [
    info["tableau_field_name"]
    for info in INDICATOR_INFO_VI.values()
    if info["tableau_field_name"] in df_wide_for_tableau.columns
]

df_wide_for_tableau = df_wide_for_tableau[base_cols + indicator_wide_cols]

print("--- Kích thước bảng wide cho Tableau")
print(df_wide_for_tableau.shape)

print("\n--- 10 dòng đầu")
display(df_wide_for_tableau.head(10))

print("\n--- Tỉ lệ missing của các cột chỉ số trong bảng wide")
wide_missing = (
    df_wide_for_tableau[indicator_wide_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)

wide_missing.columns = ["indicator_field", "missing_pct"]
display(wide_missing)

## **8. KIỂM TRA GIÁ TRỊ BẤT THƯỜNG**

### **8.1. Kiểm tra min-max theo từng chỉ số**

Với dữ liệu WDI, nhiều giá trị cực trị không nhất thiết là lỗi. Ví dụ:

* GDP bình quân đầu người rất cao ở một số nền kinh tế nhỏ hoặc nhóm thu nhập cao.
* Phát thải CO2 bình quân đầu người có thể rất cao ở các quốc gia khai thác dầu khí.
* Tỷ lệ nhập học gross có thể vượt 100% vì chỉ số này tính tổng số học sinh nhập học bất kể tuổi.
* Tăng trưởng GDP có thể âm mạnh trong khủng hoảng hoặc dương rất cao khi nền kinh tế phục hồi từ nền thấp.

Do đó, nhóm không loại ngoại lai bằng IQR một cách máy móc. Thay vào đó, nhóm thống kê min-max để phục vụ kiểm tra và diễn giải trong báo cáo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 8: THỐNG KÊ MIN-MAX VÀ KIỂM TRA GIÁ TRỊ BẤT THƯỜNG
# ─────────────────────────────────────────────────────────────

indicator_value_summary = (
    df_long_clean
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "unit_vi"
    ])
    .agg(
        count=("value", "count"),
        min_value=("value", "min"),
        q1=("value", lambda s: s.quantile(0.25)),
        median=("value", "median"),
        mean=("value", "mean"),
        q3=("value", lambda s: s.quantile(0.75)),
        max_value=("value", "max")
    )
    .reset_index()
)

numeric_cols = ["min_value", "q1", "median", "mean", "q3", "max_value"]
indicator_value_summary[numeric_cols] = indicator_value_summary[numeric_cols].round(3)

print("--- Thống kê giá trị theo từng chỉ số")
display(indicator_value_summary)

# Lấy top/bottom 5 giá trị theo từng chỉ số để kiểm tra nhanh
extreme_records = []

for code in df_long_clean["indicator_code"].unique():
    temp = df_long_clean[df_long_clean["indicator_code"] == code].copy()
    
    bottom = temp.nsmallest(5, "value")
    bottom["extreme_type"] = "Nhỏ nhất"
    
    top = temp.nlargest(5, "value")
    top["extreme_type"] = "Lớn nhất"
    
    extreme_records.append(pd.concat([bottom, top], ignore_index=True))

df_extreme_values = pd.concat(extreme_records, ignore_index=True)

df_extreme_values = df_extreme_values[
    [
        "indicator_code",
        "indicator_name_vi",
        "extreme_type",
        "country_name",
        "country_code",
        "country_type",
        "year",
        "value",
        "unit_vi"
    ]
].sort_values(["indicator_code", "extreme_type", "value"])

print("\n--- Top/bottom giá trị để kiểm tra bất thường")
display(df_extreme_values.head(60))

## **9. TẠO CÁC BẢNG PHỤ TRỢ CHO DASHBOARD**

### **9.1. Bảng danh mục quốc gia và bảng danh mục chỉ số**

Để thiết kế dashboard trong Tableau gọn hơn, nhóm tạo thêm các bảng phụ trợ:

* `country_dimension`: danh sách quốc gia/vùng/tổng hợp.
* `indicator_dimension`: danh sách chỉ số, tên tiếng Việt, nhóm chỉ số và đơn vị đo.
* `data_quality_indicator`: tóm tắt chất lượng dữ liệu theo chỉ số.
* `data_quality_year`: tóm tắt chất lượng dữ liệu theo năm.

Các bảng này có thể dùng để viết báo cáo hoặc connect thêm vào Tableau nếu cần.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 9: TẠO DIMENSION TABLES VÀ BẢNG CHẤT LƯỢNG DỮ LIỆU
# ─────────────────────────────────────────────────────────────

country_dimension = (
    df_long[[
        "country_name",
        "country_code",
        "country_long_name",
        "country_short_name",
        "country_type",
        "is_aggregate",
        "region",
        "income_group",
        "currency_unit"
    ]]
    .drop_duplicates()
    .sort_values(["country_type", "region", "income_group", "country_name"])
    .reset_index(drop=True)
)

indicator_dimension = (
    indicator_metadata[[
        "indicator_code",
        "indicator_name_en",
        "indicator_name_vi",
        "indicator_group",
        "unit_vi",
        "short_name",
        "tableau_field_name"
    ]]
    .drop_duplicates()
    .sort_values(["indicator_group", "indicator_code"])
    .reset_index(drop=True)
)

data_quality_indicator = missing_by_indicator.copy()
data_quality_year = missing_by_year.copy()
data_quality_indicator_year = missing_by_indicator_year.copy()

print("--- Country dimension")
display(country_dimension.head(20))

print("\n--- Indicator dimension")
display(indicator_dimension)

print("\n--- Data quality by indicator")
display(data_quality_indicator)

## **10. XUẤT DỮ LIỆU SAU TIỀN XỬ LÝ**

### **10.1. Các file đầu ra dùng cho Tableau**

Sau khi tiền xử lý, nhóm chỉ xuất các file dữ liệu cần thiết để kết nối vào Tableau và xây dựng dashboard.

| File | Mục đích |
| :--- | :--- |
| `wdi_long_clean.csv` | File chính dùng để vẽ biểu đồ trong Tableau, dạng long, đã loại bỏ giá trị thiếu |
| `wdi_wide_for_tableau.csv` | File phụ dùng cho các biểu đồ so sánh/tương quan giữa nhiều chỉ số |

File khuyến nghị dùng chính trong Tableau là `wdi_long_clean.csv`.

File `wdi_wide_for_tableau.csv` chỉ dùng khi cần vẽ các biểu đồ như scatter plot giữa GDP bình quân đầu người và tuổi thọ, chi tiêu y tế và tử vong trẻ em, hoặc CO2 và năng lượng tái tạo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 10: XUẤT FILE CSV DÙNG CHO TABLEAU
# ─────────────────────────────────────────────────────────────

output_files = {
    "wdi_long_clean.csv": df_long_clean,
    "wdi_wide_for_tableau.csv": df_wide_for_tableau
}

for filename, df_out in output_files.items():
    output_path = OUTPUT_DIR / filename
    df_out.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Đã xuất: {output_path} | shape = {df_out.shape}")

print("\nHoàn tất xuất dữ liệu dùng cho Tableau.")

## **11. TỔNG KẾT QUY TRÌNH TIỀN XỬ LÝ**

Quy trình tiền xử lý đã hoàn thành các bước chính sau:

1. Đọc dữ liệu chính từ `Data.csv`.
2. Đọc metadata sạch từ file `Metadata.xlsx`, gồm metadata chỉ số và metadata quốc gia/vùng lãnh thổ.
3. Loại bỏ các dòng footer và dòng trống không phải quan sát dữ liệu.
4. Chuẩn hóa ký hiệu missing `..` thành `NaN`.
5. Ép kiểu dữ liệu các cột năm sang dạng số.
6. Chuẩn hóa metadata chỉ số từ sheet `Series - Metadata`.
7. Chuẩn hóa metadata quốc gia/vùng lãnh thổ từ sheet `Country - Metadata`.
8. Việt hóa tên chỉ số và phân nhóm chỉ số theo các chủ đề: kinh tế, sức khỏe, môi trường, giáo dục.
9. Chuyển dữ liệu từ dạng rộng sang dạng dài để phù hợp với Tableau.
10. Bổ sung thông tin khu vực, nhóm thu nhập và đơn vị tiền tệ cho từng quốc gia/vùng lãnh thổ.
11. Đánh dấu các nhóm tổng hợp của World Bank bằng cột `is_aggregate`.
12. Thống kê chất lượng dữ liệu theo chỉ số, theo năm và theo quốc gia.
13. Tạo bảng long phục vụ dashboard, trực quan hóa.

### **Lưu ý khi phân tích**

* Không nên tự ý điền giá trị thiếu cho dữ liệu WDI vì có thể làm sai lệch xu hướng phát triển.
* Khi phân tích tỷ lệ nghèo, cần chú ý vì chỉ số này thường thiếu nhiều dữ liệu hơn các chỉ số còn lại.
* Khi xếp hạng quốc gia, nên lọc `country_type = "Quốc gia/vùng lãnh thổ"` để loại các nhóm tổng hợp như `World`, `High income`, `OECD members`.
* Khi cần so sánh với trung bình khu vực hoặc thế giới, có thể giữ lại `country_type = "Nhóm tổng hợp/khu vực"`.
* Có thể dùng thêm `region` và `income_group` làm filter trong Tableau để dashboard chuyên nghiệp hơn.
* File chính nên dùng trong Tableau là `wdi_long_clean.csv`.